# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. `5 (2, 3)` - массив `a` равен `[[0,1,2][3,4,5]]`. В срезе значение по индексу 1 равно `[3,4,5]`, далее внутри этого массива значение по индексу 2 равно `5`, а атрибут `shape` возвращает кортеж `(2, 3)`, который указывает на размер массива.
2. `1 8` - `argmax()` возвращает первый индекс максимального значения из массива, а `max()` возвращает само это макс. значение. 
3. `99` - массивы `a` и `b` указывают на одни и те же ячейки памяти, поэтому изменение элемента `b[0]` влияет на `a[0]`.
4. `2 0.5` - выражение `(a > 3)` дает массив булевых значений, сравнивая каждый элемент массива `a` с условием: `[False, True, True, False]`. Метод `sum()` суммирует все элементы (False=0 и True=1), поэтому: `2`. `mean()` находит среднее (суммирует результаты `(a > 3)`: F+T+T+F=2, затем делит сумму на количество элементов массива: 2 / 4 = 0.5)
5. `(2, 3) 0.25` - `np.zeros((2,3)).shape` дает кортеж размерности (2, 3) для массива нулей: `[[0,0,0],[0,0,0]]`. Функция `linspace()` возвращает массив 5 равномерно распределённых чисел от 0 до 1: `[0, 0.25, 5, 0.75, 1]`. Здесь элемент с индексом `[1]` равен 0.25.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [4]:
# База

max_ord_idx = amount.argmax()

print("1. Самый дорогой заказ")
print(f"id: {max_ord_idx}")
print(f"cумма: {amount[max_ord_idx]}")
print(f"клиент: {cust[max_ord_idx]}")
print(f"город: {city[max_ord_idx]}")

revenue_paid = amount[paid].sum()
sum_unpaid = amount[~paid].sum()

print(f"\n2. Выручка оплаченных заказов: {revenue_paid}")
print(f"Сумма неоплаченных заказов: {sum_unpaid}")

shymkent_count = (city == "Шымкент").sum()
total_orders = len(amount)
shymkent_order_pct = (shymkent_count / total_orders) * 100

print(f"\n3. Заказы в Шымкенте")
print(f"Количество заказов: {shymkent_count}")
print(f"Доля от всех заказов: {shymkent_order_pct:.2f}%")

1. Самый дорогой заказ
id: 1001
cумма: 9999990.0
клиент: TEST
город: Алматы

2. Выручка оплаченных заказов: 101909360.0
Сумма неоплаченных заказов: 7974000.0

3. Заказы в Шымкенте
Количество заказов: 804
Доля от всех заказов: 14.72%


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [9]:
# Стандарт
# Все оплаченный заказы дороже 50 000
discount_mask = paid & (amount > 50000)
count_all = len(amount[discount_mask])
discount_all = amount[discount_mask].sum() * 0.15

# Все оплаченный заказы без клиента TEST
no_test = discount_mask & (cust != "TEST")
count_no_test = len(amount[no_test])
discount_no_test = amount[no_test].sum() * 0.15

print(f"4\nКоличество заказов: {count_all} на сумму: {amount[discount_mask].sum()}")
print("Недополученная выручка:", discount_all)
print(f"Количество заказов без TEST: {count_no_test} на сумму: {amount[no_test].sum()}")
print(f"Недополученная выручка без TEST: {discount_no_test}")

diff = discount_all / discount_no_test
diff_pct = (discount_all - discount_no_test) / discount_all * 100

print(f"\nПочему ответы различаются почти {round(diff, 1)} раза:")
print(f"Потому что TEST имеет {count_all - count_no_test} заказов на сумму {amount[discount_mask].sum() - amount[no_test].sum()}")
print(f"Это примерно {round(diff_pct)}% от общей суммы заказов")

print("\n5. Для каждого города — доля заказов, доставленных за 1–2 дня:")
for c in np.unique(city):
    city_mask = city == c
    
    fast_mask = city_mask & (days >= 1) & (days <= 2)

    total = city_mask.sum()
    fast = fast_mask.sum()

    share = fast / total * 100

    print(c, ":", fast, "/", total, "=", round(share, 2), "%")
    
avg = amount[paid].mean()
print("\n6. Средний чек:", round(avg, 2))

4
Количество заказов: 127 на сумму: 38016150.0
Недополученная выручка: 5702422.5
Количество заказов без TEST: 124 на сумму: 8016180.0
Недополученная выручка без TEST: 1202427.0

Почему ответы различаются почти 4.7 раза:
Потому что TEST имеет 3 заказов на сумму 29999970.0
Это примерно 79% от общей суммы заказов

5. Для каждого города — доля заказов, доставленных за 1–2 дня:
Актобе : 247 / 397 = 62.22 %
Алматы : 1412 / 2188 = 64.53 %
Астана : 883 / 1390 = 63.53 %
Атырау : 94 / 146 = 64.38 %
Караганда : 365 / 538 = 67.84 %
Шымкент : 240 / 804 = 29.85 %

6. Средний чек: 20780.86


## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [12]:
# Челлендж
import time

start = time.time()

total_for = 0

for i in range(len(amount)):
    if paid[i]:
        total_for += amount[i]
        
time_for = time.time() - start

print(f"Сумма: {total_for}")
print(f"Время for: {time_for}")

start = time.time()

total_numpy = amount[paid].sum()

time_numpy = time.time() - start

print("Сумма:", total_numpy)
print("Время NumPy:", time_numpy)

speedup = time_for / time_numpy

print("Ускорение:", round(speedup, 2), "раз")

Сумма: 101909360.0
Время for: 0.006930112838745117
Сумма: 101909360.0
Время NumPy: 0.004746198654174805
Ускорение: 1.46 раз


In [13]:
# 
np.random.seed(42)

A = np.random.rand(1000) < 0.05
B = np.random.rand(1000) < 0.06

print("Покупки A:", A.sum())
print("Покупки B:", B.sum())

# Повторяем эксперимент 100 раз
wins = 0

for i in range(100):
    A = np.random.rand(1000) < 0.05
    B = np.random.rand(1000) < 0.06

    purchases_A = A.sum()
    purchases_B = B.sum()

    if purchases_B > purchases_A:
        wins += 1

print("B лучше A:", wins, "раз")
print("Доля побед B:", wins / 100 * 100, "%")

Покупки A: 54
Покупки B: 58
B лучше A: 84 раз
Доля побед B: 84.0 %
